In [0]:
df = spark.read.format("binaryFile") \
    .load("/Volumes/bi/default/bi/Przykladowa Faktura Usluga.pdf")

from pyspark.sql.functions import expr

df_parsed = df.withColumn(
    "parsed",
    expr("ai_parse_document(content, map('mode', 'markdown'))")
)

display(df_parsed.select("path", "parsed"))

In [0]:
from pyspark.sql.functions import col, expr, concat_ws

df_text = df_parsed.selectExpr(
    "path",
    """
    concat_ws('\\n',
        transform(
            try_cast(parsed:document:elements AS ARRAY<VARIANT>),
            e -> try_cast(e:content AS STRING)
        )
    ) AS invoice_text
    """,
    "parsed:error_status AS error_status"
)

display(df_text)

In [0]:
df_text.write.mode("overwrite").saveAsTable("bi.default.bronze_invoices_ocr")

In [0]:
from pyspark.sql.functions import regexp_extract, current_timestamp, col

df_invoice = df_text.withColumn(
    "invoice_number",
    regexp_extract(col("invoice_text"),
                   r"FAKTURA VAT NR\\s+([A-Z0-9/.-]+)", 1)
).withColumn(
    "issue_date",
    regexp_extract(col("invoice_text"),
                   r"Data wystawienia:\\s*([0-9]{2}\\.[0-9]{2}\\.[0-9]{4})", 1)
).withColumn(
    "sale_date",
    regexp_extract(col("invoice_text"),
                   r"Data sprzedaży:\\s*([0-9]{2}\\.[0-9]{2}\\.[0-9]{4})", 1)
).withColumn(
    "seller",
    regexp_extract(col("invoice_text"),
                   r"Sprzedawca\\s*\\n([^\\n]+)", 1)
).withColumn(
    "buyer",
    regexp_extract(col("invoice_text"),
                   r"Nabywca\\s*\\n([^\\n]+)", 1)
).withColumn(
    "payment_method",
    regexp_extract(col("invoice_text"),
                   r"Forma płatności:\\s*([^\\n]+)", 1)
).withColumn(
    "bank_account",
    regexp_extract(col("invoice_text"),
                   r"Numer rachunku:\\s*([0-9 ]+)", 1)
).withColumn(
    "net_amount",
    regexp_extract(col("invoice_text"),
                   r"Wartość netto:\\s*([0-9\\s,]+)", 1)
).withColumn(
    "vat_amount",
    regexp_extract(col("invoice_text"),
                   r"VAT 23%:\\s*([0-9\\s,]+)", 1)
).withColumn(
    "gross_amount",
    regexp_extract(col("invoice_text"),
                   r"Do zapłaty brutto:\\s*([0-9\\s,]+)", 1)
).withColumn(
    "load_timestamp",
    current_timestamp()
)

display(df_invoice)

In [0]:
df_invoice.select("invoice_text").collect()[0][0]

In [0]:
from pyspark.sql.functions import regexp_extract, current_timestamp, col

df_silver_invoice = df_text.select(
    "path",

    regexp_extract(
        col("invoice_text"),
        r"FAKTURA VAT NR\s+([A-Z0-9/.-]+)",
        1
    ).alias("invoice_number"),

    regexp_extract(
        col("invoice_text"),
        r"Sprzedawca\s*\n([^\n]+)",
        1
    ).alias("seller_name"),

    regexp_extract(
        col("invoice_text"),
        r"Nabywca\s*\n([^\n]+)",
        1
    ).alias("buyer_name"),

    regexp_extract(
        col("invoice_text"),
        r"Data wystawienia:</td><td>([0-9]{2}\.[0-9]{2}\.[0-9]{4})",
        1
    ).alias("issue_date"),

    regexp_extract(
        col("invoice_text"),
        r"Forma płatności:</td><td>([^<]+)",
        1
    ).alias("payment_method"),

    regexp_extract(
        col("invoice_text"),
        r"Do zapłaty brutto:\s*([0-9\s,]+)",
        1
    ).alias("gross_total"),

    current_timestamp().alias("load_timestamp")
)

display(df_silver_invoice)

In [0]:
df_silver_invoice.write \
    .mode("overwrite") \
    .saveAsTable("bi.default.silver_invoices")

In [0]:
from pyspark.sql.functions import col, current_timestamp, regexp_extract, lit, monotonically_increasing_id

df_text = spark.table("bi.default.bronze_invoices_ocr")

df_transactions = df_text.select(
    monotonically_increasing_id().alias("transaction_id"),
    col("path").alias("source_file"),
    regexp_extract(col("invoice_text"), r"FAKTURA VAT NR\\s+([A-Z0-9/.-]+)", 1).alias("invoice_number"),
    regexp_extract(col("invoice_text"), r"Data wystawienia:\\s*([0-9]{2}\\.[0-9]{2}\\.[0-9]{4})", 1).alias("issue_date"),
    regexp_extract(col("invoice_text"), r"Do zapłaty brutto:\\s*([0-9\\s,]+)", 1).alias("gross_amount"),
    current_timestamp().alias("load_timestamp")
).dropDuplicates(["invoice_number"])

df_customers = df_text.select(
    regexp_extract(col("invoice_text"), r"Nabywca\\s*\\n([^\\n]+)", 1).alias("customer_name")
).filter(col("customer_name") != "").dropDuplicates()

df_products = df_text.select(
    lit("usługa_z_faktury").alias("product_name")
).dropDuplicates()

df_transactions.write.mode("overwrite").format("delta").saveAsTable("bi.default.silver_transactions")
df_customers.write.mode("overwrite").format("delta").saveAsTable("bi.default.silver_customers")
df_products.write.mode("overwrite").format("delta").saveAsTable("bi.default.silver_products")

In [0]:
from pyspark.sql.functions import sum, count, col, to_date, date_format, round


df_trans = spark.table("bi.default.silver_transactions")
df_cust  = spark.table("bi.default.silver_customers")


df_gold_sales = df_trans.select(
    col("invoice_number"),
    col("issue_date"),
    col("gross_amount"),
    col("source_file")
).filter(col("invoice_number") != "")

df_gold_sales.write.mode("overwrite").format("delta") \
    .saveAsTable("bi.default.gold_invoice_summary")


df_gold_kpi = df_trans.agg(
    count("invoice_number").alias("total_invoices"),
    sum("gross_amount").alias("total_gross_amount_pln")
)

df_gold_kpi.write.mode("overwrite").format("delta") \
    .saveAsTable("bi.default.gold_kpi_summary")

df_gold_by_customer = df_trans.crossJoin(df_cust.limit(1)).select(
    col("customer_name"),
    col("invoice_number"),
    col("gross_amount"),
    col("issue_date")
)

df_gold_by_customer.write.mode("overwrite").format("delta") \
    .saveAsTable("bi.default.gold_customer_transactions")

display(df_gold_kpi)
display(df_gold_sales)

---------------------------------------------------------------------------
NumberFormatException                     Traceback (most recent call last)
File <command-8620012963450995>, line 25
     18 # --- GOLD 2: Liczba faktur i suma brutto (top-level KPI) ---
     19 df_gold_kpi = df_trans.agg(
     20     count("invoice_number").alias("total_invoices"),
     21     sum("gross_amount").alias("total_gross_amount_pln")
     22 )
     24 df_gold_kpi.write.mode("overwrite").format("delta") \
---> 25     .saveAsTable("bi.default.gold_kpi_summary")
     27 # --- GOLD 3: Zestawienie transakcji z klientami (JOIN) ---
     28 df_gold_by_customer = df_trans.crossJoin(df_cust.limit(1)).select(
     29     col("customer_name"),
     30     col("invoice_number"),
     31     col("gross_amount"),
     32     col("issue_date")
     33 )

File /databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/readwriter.py:737, in DataFrameWriter.saveAsTable(self, name, format, mode, partitionBy, 